# DEE — Test crítico espectral: T³ vs S³ a N=500

**Pregunta:** después del MC con annealing completo, ¿el espectro del Laplaciano emergente discrimina T³ de S³?

**Por qué este test es definitivo:** los observables locales (var(κ), ⟨I⟩) del experimento anterior no discriminaron las topologías. Pero SIM 15 mostró discriminación 13.4σ con observables globales (espectro completo). Acá medimos el espectro de las configuraciones MC finales para ver si el sistema dinámico también discrimina globalmente.

**Configuración:**
- N = 500 nodos
- 1 corrida por topología (T³ + S³), ambas con F = var(κ) + α·iso
- Annealing completo: 25500 pasos por topología
- Inicialización: uniforme aleatoria
- T objetivo: 0 (mínimo de F)

**Comparación final:** espectro de las 2 corridas MC contra benchmarks T³ y S³ ideales con jitter pequeño.

**Veredicto:**
- Si espectro_MC(T³) ~ benchmark_T³ y espectro_MC(S³) ~ benchmark_S³ → discriminación dinámica positiva
- Si los dos espectros MC son indistinguibles → el modelo no discrimina topología dinámicamente

---

In [ ]:
import os
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix, diags
from scipy.linalg import eigh
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import time

PLOT_DIR = 'plots_spec'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name):
    plt.savefig(os.path.join(PLOT_DIR, f'{name}.png'), dpi=120, bbox_inches='tight')
    print(f'  -> {name}.png')

# Configuración global
THETA_D = 43.4
N_TARGET = 500
K = 20
L_T3 = 1.0
R_S3 = (1.0 / (2 * np.pi**2))**(1/3)  # ~0.366

print(f'N = {N_TARGET}, k = {K}')
print(f'T³: L = {L_T3}')
print(f'S³: R = {R_S3:.4f}, volumen = {2*np.pi**2*R_S3**3:.4f}')

In [ ]:
# Inicializaciones

def init_uniform_T3(N_target, seed=42, L=1.0):
    rng = np.random.default_rng(seed)
    return rng.uniform(0, L, size=(N_target, 3)), N_target

def init_uniform_S3(N_target, seed=42, R=R_S3):
    rng = np.random.default_rng(seed)
    pts = rng.normal(size=(N_target, 4))
    norms = np.linalg.norm(pts, axis=1, keepdims=True)
    return pts / norms * R, N_target

# Grids ideales para benchmarks
def grid_T3_ideal(N_target, L=1.0, jitter=0.05, seed=999):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    spacing = L / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter*spacing, jitter*spacing, size=points.shape)
    points = points % L
    return points, n_side**3

def grid_S3_ideal(N_target, R=R_S3, jitter=0.05, seed=999):
    """Fibonacci-like sampling sobre S^3 con perturbacion pequeña."""
    rng = np.random.default_rng(seed)
    pts = []
    phi1 = (np.sqrt(5) - 1) / 2
    phi2 = phi1**2
    for i in range(N_target):
        s = (i + 0.5) / N_target
        u1 = (i * phi1) % 1
        u2 = (i * phi2) % 1
        theta = 2 * np.pi * u1
        phi = 2 * np.pi * u2
        eta = np.arccos(1 - 2*s)
        x = R * np.array([
            np.cos(eta),
            np.sin(eta) * np.cos(theta),
            np.sin(eta) * np.sin(theta) * np.cos(phi),
            np.sin(eta) * np.sin(theta) * np.sin(phi),
        ])
        pts.append(x)
    pts = np.array(pts)
    spacing = 2 * R * (np.pi**2 * 2 / N_target)**(1/3)
    perturb = rng.normal(0, spacing*jitter, size=(N_target, 4))
    pts_p = pts + perturb
    pts_p = pts_p / np.linalg.norm(pts_p, axis=1, keepdims=True) * R
    return pts_p, N_target

print('Inicializaciones listas')

In [ ]:
# Vecindarios y Laplaciano

def neighbors_T3_fast(points, k=20, L=1.0):
    """k-NN con imágenes periodicas usando cKDTree."""
    N = len(points)
    images = []
    image_to_orig = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:
                shift = np.array([dx*L, dy*L, dz*L])
                images.append(points + shift)
                image_to_orig.append(np.arange(N))
    images_all = np.concatenate(images, axis=0)
    image_to_orig_all = np.concatenate(image_to_orig)
    tree = cKDTree(images_all)
    
    nbrs = np.zeros((N, k), dtype=int)
    nbr_dists = np.zeros((N, k))
    nbr_pos = np.zeros((N, k, 3))
    
    for i in range(N):
        dists, idxs = tree.query(points[i], k=k+1)
        valid = ~((image_to_orig_all[idxs] == i) & (dists < 1e-12))
        v_idx = np.where(valid)[0][:k]
        nbrs[i] = image_to_orig_all[idxs[v_idx]]
        nbr_dists[i] = dists[v_idx]
        nbr_pos[i] = images_all[idxs[v_idx]]
    return nbrs, nbr_dists, nbr_pos

def neighbors_S3_fast(points, k=20, R=R_S3):
    """k-NN con distancia geodesica vectorizada."""
    N = len(points)
    cos_a = np.clip(points @ points.T / R**2, -1, 1)
    np.fill_diagonal(cos_a, 1.0 - 1e-12)
    geod = R * np.arccos(cos_a)
    np.fill_diagonal(geod, np.inf)
    nbrs = np.argsort(geod, axis=1)[:, :k]
    nbr_dists = np.take_along_axis(geod, nbrs, axis=1)
    nbr_pos = points[nbrs]
    return nbrs, nbr_dists, nbr_pos

def laplacian_from_neighbors(nbrs, nbr_dists, alpha=2.0):
    """Construir Laplaciano esparcido desde lista de vecinos."""
    N = len(nbrs)
    k = nbrs.shape[1]
    rows = np.repeat(np.arange(N), k)
    cols = nbrs.ravel()
    weights = (1.0 / nbr_dists**alpha).ravel()
    W = csr_matrix((weights, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    deg = np.array(W.sum(axis=1)).flatten()
    return diags(deg) - W, deg

print('Funciones Laplaciano listas')

In [ ]:
# F vectorizado (rapido)

def compute_F_vec(points, neighbor_fn, alpha_iso, alpha_kernel=2.0, **kwargs):
    """
    F = N * var(kappa) + alpha_iso * sum (1 - I_i)^2
    Vectorizado para velocidad.
    """
    nbrs, nbr_dists, nbr_pos = neighbor_fn(points, **kwargs)
    N = len(points)
    
    # kappa
    deg = np.sum(1.0 / nbr_dists**alpha_kernel, axis=1)
    kappa = deg / np.mean(deg)
    F_var = N * np.var(kappa)
    
    # Isotropia vectorizada
    if alpha_iso > 0:
        # Vector posicion relativa: nbr_pos[i,j,:] - points[i,:]
        diffs = nbr_pos - points[:, None, :]  # (N, k, dim)
        # Tensor de momento: M_i = (1/k) sum_j diffs[i,j] outer diffs[i,j]
        M = np.einsum('ikj,ikm->ijm', diffs, diffs) / nbrs.shape[1]  # (N, dim, dim)
        # Autovalores
        eigvals_all = np.linalg.eigvalsh(M)  # (N, dim)
        eigvals_all = np.maximum(eigvals_all, 1e-12)
        # Tomar los 3 mayores autovalores
        if eigvals_all.shape[1] > 3:
            eigvals_top = eigvals_all[:, -3:]
        else:
            eigvals_top = eigvals_all
        geom = np.exp(np.mean(np.log(eigvals_top), axis=1))
        arith = np.mean(eigvals_top, axis=1)
        iso = geom / arith
        F_iso = np.sum((1.0 - iso)**2)
        F_total = F_var + alpha_iso * F_iso
    else:
        iso = None
        F_total = F_var
        F_iso = 0
    
    return F_total, F_var, F_iso, kappa, iso, nbrs, nbr_dists

# Test rapido
pts_t3, _ = init_uniform_T3(N_TARGET)
t0 = time.time()
F, F_v, F_i, k_, iso_, _, _ = compute_F_vec(pts_t3, neighbors_T3_fast, alpha_iso=1.0, k=K, L=L_T3)
tT3 = time.time() - t0
print(f'T³ N={N_TARGET}: F={F:.2f} (var={F_v:.2f}, iso={F_i:.2f}), tiempo: {tT3*1000:.0f} ms')

pts_s3, _ = init_uniform_S3(N_TARGET)
t0 = time.time()
F, F_v, F_i, k_, iso_, _, _ = compute_F_vec(pts_s3, neighbors_S3_fast, alpha_iso=1.0, k=K, R=R_S3)
tS3 = time.time() - t0
print(f'S³ N={N_TARGET}: F={F:.2f} (var={F_v:.2f}, iso={F_i:.2f}), tiempo: {tS3*1000:.0f} ms')

print(f'\nEstimacion 25500 pasos:')
print(f'  T³: ~{tT3*25500/60:.0f} min')
print(f'  S³: ~{tS3*25500/60:.0f} min')

In [ ]:
# Calibracion de alpha_iso
_, F_var_T3_r, F_iso_T3_r, _, _, _, _ = compute_F_vec(
    pts_t3, neighbors_T3_fast, alpha_iso=1.0, k=K, L=L_T3
)
ALPHA_ISO_T3 = F_var_T3_r / max(F_iso_T3_r, 1e-6)

_, F_var_S3_r, F_iso_S3_r, _, _, _, _ = compute_F_vec(
    pts_s3, neighbors_S3_fast, alpha_iso=1.0, k=K, R=R_S3
)
ALPHA_ISO_S3 = F_var_S3_r / max(F_iso_S3_r, 1e-6)

print(f'α_iso (T³) = {ALPHA_ISO_T3:.4f}')
print(f'α_iso (S³) = {ALPHA_ISO_S3:.4f}')

In [ ]:
# MC con annealing

def gen_move_T3(rng, sigma):
    return rng.normal(0, sigma, size=3)

def gen_move_S3(rng, sigma, current_pos, R=R_S3):
    v = rng.normal(size=4)
    radial = (v @ current_pos / R**2) * current_pos
    tangent = v - radial
    nrm = np.linalg.norm(tangent)
    if nrm > 1e-10:
        tangent = tangent / nrm
    return tangent * rng.normal(0, sigma)

def move_T3(points, i, delta, L=1.0):
    new_pts = points.copy()
    new_pts[i] = (points[i] + delta) % L
    return new_pts

def move_S3(points, i, delta, R=R_S3):
    new_pts = points.copy()
    new = points[i] + delta
    new_pts[i] = new / np.linalg.norm(new) * R
    return new_pts

def run_mc_annealing(pts_init, T_schedule, n_steps_per_T, topology,
                     alpha_iso, sigma_move=0.02, seed=42, **kwargs):
    """MC annealing con N grande. Recalcula F entero (vectorizado)."""
    if topology == 'T3':
        neighbor_fn = neighbors_T3_fast
        move_fn = move_T3
        gen_fn = gen_move_T3
    else:
        neighbor_fn = neighbors_S3_fast
        move_fn = move_S3
        gen_fn = gen_move_S3
    
    rng = np.random.default_rng(seed)
    points = pts_init.copy()
    F_curr, _, _, _, _, _, _ = compute_F_vec(points, neighbor_fn, alpha_iso, **kwargs)
    
    history = {'step': [], 'T': [], 'F': [], 'kappa_var': [], 'iso_mean': []}
    total_step = 0
    t_start = time.time()
    
    for T in T_schedule:
        n_acc = 0
        for s in range(n_steps_per_T):
            i = rng.integers(len(points))
            if topology == 'T3':
                delta = gen_fn(rng, sigma_move)
                pts_new = move_fn(points, i, delta, L=kwargs['L'])
            else:
                delta = gen_fn(rng, sigma_move, points[i], R=kwargs['R'])
                pts_new = move_fn(points, i, delta, R=kwargs['R'])
            
            F_new, _, _, _, _, _, _ = compute_F_vec(pts_new, neighbor_fn, alpha_iso, **kwargs)
            dF = F_new - F_curr
            
            if T > 0:
                accept = (dF < 0) or (rng.random() < np.exp(-dF / T))
            else:
                accept = (dF < 0)
            
            if accept:
                points = pts_new
                F_curr = F_new
                n_acc += 1
            
            if total_step % 1000 == 0:
                _, _, _, k_, iso_, _, _ = compute_F_vec(points, neighbor_fn, max(alpha_iso, 1.0), **kwargs)
                history['step'].append(total_step)
                history['T'].append(T)
                history['F'].append(F_curr)
                history['kappa_var'].append(k_.var())
                history['iso_mean'].append(iso_.mean() if iso_ is not None else 0)
            total_step += 1
        
        ar = n_acc / n_steps_per_T
        elapsed = time.time() - t_start
        print(f'    T={T:6.2f}  acept={ar*100:5.1f}%  F={F_curr:.2f}  t={elapsed:.0f}s')
    
    return points, F_curr, history

print('MC listo')

---
## Fase A — MC sobre T³

In [ ]:
# Schedule de annealing: tres fases hot, mid, target
# 25500 pasos totales: 7500 hot + 7500 mid + 4500 target_intermediate + 6000 final
T_schedule_full = [THETA_D, THETA_D/3, THETA_D/10, 0.0]
n_steps_T3 = [7500, 7500, 4500, 6000]  # total = 25500

print(f'Annealing T³: total {sum(n_steps_T3)} pasos')
print(f'  Schedule: {T_schedule_full}')
print(f'  Pasos por fase: {n_steps_T3}\n')

pts_init_T3, N_T3 = init_uniform_T3(N_TARGET, seed=42, L=L_T3)
print(f'Estado inicial T³: N = {N_T3}')

pts_T3_final = pts_init_T3.copy()
history_T3_total = {'step': [], 'T': [], 'F': [], 'kappa_var': [], 'iso_mean': []}
offset = 0
t_start = time.time()

for T_phase, n_steps in zip(T_schedule_full, n_steps_T3):
    print(f'\n  Fase T = {T_phase:.2f}, {n_steps} pasos:')
    pts_T3_final, _, hist = run_mc_annealing(
        pts_T3_final, [T_phase], n_steps, topology='T3',
        alpha_iso=ALPHA_ISO_T3, sigma_move=0.02, seed=42 + offset,
        k=K, L=L_T3
    )
    for kk in hist:
        if kk == 'step':
            history_T3_total[kk].extend([s + offset for s in hist[kk]])
        else:
            history_T3_total[kk].extend(hist[kk])
    offset += n_steps

t_T3 = time.time() - t_start
print(f'\n✓ T³ completado en {t_T3:.0f}s ({t_T3/60:.1f} min)')

# Mediciones finales
F_T3, F_var_T3, F_iso_T3, kappa_T3, iso_T3, _, _ = compute_F_vec(
    pts_T3_final, neighbors_T3_fast, max(ALPHA_ISO_T3, 1.0), k=K, L=L_T3
)
print(f'  F final = {F_T3:.2f}')
print(f'  var(κ) = {kappa_T3.var():.5f}')
print(f'  ⟨I⟩ = {iso_T3.mean():.4f}')

---
## Fase B — MC sobre S³

In [ ]:
n_steps_S3 = [7500, 7500, 4500, 6000]  # mismo schedule

print(f'Annealing S³: total {sum(n_steps_S3)} pasos\n')

pts_init_S3, N_S3 = init_uniform_S3(N_TARGET, seed=42, R=R_S3)
print(f'Estado inicial S³: N = {N_S3}')

pts_S3_final = pts_init_S3.copy()
history_S3_total = {'step': [], 'T': [], 'F': [], 'kappa_var': [], 'iso_mean': []}
offset = 0
t_start = time.time()

for T_phase, n_steps in zip(T_schedule_full, n_steps_S3):
    print(f'\n  Fase T = {T_phase:.2f}, {n_steps} pasos:')
    pts_S3_final, _, hist = run_mc_annealing(
        pts_S3_final, [T_phase], n_steps, topology='S3',
        alpha_iso=ALPHA_ISO_S3, sigma_move=0.02, seed=42 + offset,
        k=K, R=R_S3
    )
    for kk in hist:
        if kk == 'step':
            history_S3_total[kk].extend([s + offset for s in hist[kk]])
        else:
            history_S3_total[kk].extend(hist[kk])
    offset += n_steps

t_S3 = time.time() - t_start
print(f'\n✓ S³ completado en {t_S3:.0f}s ({t_S3/60:.1f} min)')

F_S3, F_var_S3, F_iso_S3, kappa_S3, iso_S3, _, _ = compute_F_vec(
    pts_S3_final, neighbors_S3_fast, max(ALPHA_ISO_S3, 1.0), k=K, R=R_S3
)
print(f'  F final = {F_S3:.2f}')
print(f'  var(κ) = {kappa_S3.var():.5f}')
print(f'  ⟨I⟩ = {iso_S3.mean():.4f}')

---
## Fase C — Análisis espectral comparativo

In [ ]:
# Calcular espectros del Laplaciano para las 4 configuraciones:
# 1. T³ MC final
# 2. S³ MC final
# 3. T³ ideal benchmark
# 4. S³ ideal benchmark

print('Computando espectros...')

def get_eigvals(points, neighbor_fn, **kwargs):
    nbrs, nbr_dists, _ = neighbor_fn(points, **kwargs)
    L_op, _ = laplacian_from_neighbors(nbrs, nbr_dists)
    eigvals = eigh(L_op.toarray(), eigvals_only=True)
    eigvals = np.sort(eigvals)
    valid = eigvals > 1e-3
    return np.sqrt(eigvals[valid])

t0 = time.time()
omega_T3_mc = get_eigvals(pts_T3_final, neighbors_T3_fast, k=K, L=L_T3)
omega_S3_mc = get_eigvals(pts_S3_final, neighbors_S3_fast, k=K, R=R_S3)

pts_T3_ideal, _ = grid_T3_ideal(N_TARGET, L=L_T3)
pts_S3_ideal, _ = grid_S3_ideal(N_TARGET, R=R_S3)
omega_T3_ideal = get_eigvals(pts_T3_ideal, neighbors_T3_fast, k=K, L=L_T3)
omega_S3_ideal = get_eigvals(pts_S3_ideal, neighbors_S3_fast, k=K, R=R_S3)

# Tambien: configuracion aleatoria como referencia 'fase desordenada'
pts_rand_T3, _ = init_uniform_T3(N_TARGET, seed=42, L=L_T3)
omega_rand_T3 = get_eigvals(pts_rand_T3, neighbors_T3_fast, k=K, L=L_T3)

print(f'Espectros calculados en {time.time()-t0:.1f}s')
print(f'\nRangos de frecuencias:')
print(f'  T³ MC:    [{omega_T3_mc.min():.3f}, {omega_T3_mc.max():.3f}], N modos: {len(omega_T3_mc)}')
print(f'  S³ MC:    [{omega_S3_mc.min():.3f}, {omega_S3_mc.max():.3f}], N modos: {len(omega_S3_mc)}')
print(f'  T³ ideal: [{omega_T3_ideal.min():.3f}, {omega_T3_ideal.max():.3f}]')
print(f'  S³ ideal: [{omega_S3_ideal.min():.3f}, {omega_S3_ideal.max():.3f}]')
print(f'  Random T³: [{omega_rand_T3.min():.3f}, {omega_rand_T3.max():.3f}]')

In [ ]:
# Plot 1: histogramas comparativos
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subplot 1: T³ MC vs T³ ideal vs random
ax = axes[0]
ax.hist(omega_rand_T3, bins=60, alpha=0.4, color='gray', label='Random inicial', density=True)
ax.hist(omega_T3_mc, bins=60, alpha=0.6, color='crimson', label='T³ MC final', density=True)
ax.hist(omega_T3_ideal, bins=60, alpha=0.5, color='steelblue', histtype='step', lw=2.5,
        label='T³ ideal benchmark', density=True)
ax.set_xlabel('ω', fontsize=12)
ax.set_ylabel('Densidad', fontsize=12)
ax.set_title('T³ — densidad de estados', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Subplot 2: S³ MC vs S³ ideal
ax = axes[1]
ax.hist(omega_S3_mc, bins=60, alpha=0.6, color='darkgreen', label='S³ MC final', density=True)
ax.hist(omega_S3_ideal, bins=60, alpha=0.5, color='purple', histtype='step', lw=2.5,
        label='S³ ideal benchmark', density=True)
ax.set_xlabel('ω', fontsize=12)
ax.set_ylabel('Densidad', fontsize=12)
ax.set_title('S³ — densidad de estados', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('120_espectros_comparacion')
plt.show()

In [ ]:
# Plot 2: comparacion directa T3 vs S3 (la pregunta crítica)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.hist(omega_T3_mc, bins=60, alpha=0.6, color='crimson', label='T³ MC final', density=True)
ax.hist(omega_S3_mc, bins=60, alpha=0.6, color='darkgreen', label='S³ MC final', density=True)
ax.set_xlabel('ω', fontsize=12)
ax.set_ylabel('Densidad', fontsize=12)
ax.set_title('TEST CRÍTICO: T³ vs S³ después de MC', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# CDF para mejor comparacion
ax = axes[1]
for omega_arr, color, label in [
    (omega_T3_mc, 'crimson', 'T³ MC'),
    (omega_S3_mc, 'darkgreen', 'S³ MC'),
    (omega_T3_ideal, 'steelblue', 'T³ ideal'),
    (omega_S3_ideal, 'purple', 'S³ ideal'),
]:
    sorted_o = np.sort(omega_arr)
    cdf = np.arange(1, len(sorted_o)+1) / len(sorted_o)
    ax.plot(sorted_o, cdf, '-', color=color, lw=2, label=label)
ax.set_xlabel('ω', fontsize=12)
ax.set_ylabel('CDF', fontsize=12)
ax.set_title('CDF comparativo (resolucion mejor)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('121_test_critico')
plt.show()

In [ ]:
# Test estadisticos cuantitativos

print('='*72)
print('TESTS ESTADÍSTICOS — Discriminación espectral T³ vs S³')
print('='*72)
print()

# Test 1: KS entre T3 MC y S3 MC
ks_mc = ks_2samp(omega_T3_mc, omega_S3_mc)
print(f'1. KS entre T³ MC y S³ MC:')
print(f'   estadistico = {ks_mc.statistic:.4f}, p-valor = {ks_mc.pvalue:.2e}')
if ks_mc.pvalue < 0.001:
    print(f'   ✓ Distribuciones SIGNIFICATIVAMENTE distintas')
else:
    print(f'   × No se distinguen estadisticamente')

# Test 2: KS entre T3 MC y T3 ideal (¿el MC se acerca al ideal?)
ks_T3 = ks_2samp(omega_T3_mc, omega_T3_ideal)
print(f'\n2. KS entre T³ MC y T³ ideal:')
print(f'   estadistico = {ks_T3.statistic:.4f}, p-valor = {ks_T3.pvalue:.2e}')

# Test 3: KS entre S3 MC y S3 ideal
ks_S3 = ks_2samp(omega_S3_mc, omega_S3_ideal)
print(f'\n3. KS entre S³ MC y S³ ideal:')
print(f'   estadistico = {ks_S3.statistic:.4f}, p-valor = {ks_S3.pvalue:.2e}')

# Test 4: KS entre T3 MC y S3 ideal (deberia ser ALTO si T3 emergio bien)
ks_T3mc_S3id = ks_2samp(omega_T3_mc, omega_S3_ideal)
print(f'\n4. KS entre T³ MC y S³ ideal (deberia ser ALTO):')
print(f'   estadistico = {ks_T3mc_S3id.statistic:.4f}, p-valor = {ks_T3mc_S3id.pvalue:.2e}')

ks_S3mc_T3id = ks_2samp(omega_S3_mc, omega_T3_ideal)
print(f'\n5. KS entre S³ MC y T³ ideal (deberia ser ALTO):')
print(f'   estadistico = {ks_S3mc_T3id.statistic:.4f}, p-valor = {ks_S3mc_T3id.pvalue:.2e}')

print()
print('Interpretacion del estadístico KS:')
print('  0 = distribuciones idénticas')
print('  1 = distribuciones completamente disjuntas')

In [ ]:
# Veredicto final: ¿hay discriminación dinámica?
print('='*72)
print('VEREDICTO ESPECTRAL')
print('='*72)
print()

# Criterio 1: T3 MC más parecido a T3 ideal que a S3 ideal
T3_correctly = ks_T3.statistic < ks_T3mc_S3id.statistic
S3_correctly = ks_S3.statistic < ks_S3mc_T3id.statistic

print(f'¿T³ MC se parece más a T³ ideal que a S³ ideal?')
print(f'   KS(T³ MC, T³ ideal) = {ks_T3.statistic:.4f}')
print(f'   KS(T³ MC, S³ ideal) = {ks_T3mc_S3id.statistic:.4f}')
print(f'   {"✓ SÍ" if T3_correctly else "✗ NO"} → T³ MC está más cerca de su benchmark correcto')
print()

print(f'¿S³ MC se parece más a S³ ideal que a T³ ideal?')
print(f'   KS(S³ MC, S³ ideal) = {ks_S3.statistic:.4f}')
print(f'   KS(S³ MC, T³ ideal) = {ks_S3mc_T3id.statistic:.4f}')
print(f'   {"✓ SÍ" if S3_correctly else "✗ NO"} → S³ MC está más cerca de su benchmark correcto')
print()

# Discriminacion T3 MC vs S3 MC
print(f'¿T³ MC y S³ MC son distinguibles entre sí?')
print(f'   KS(T³ MC, S³ MC) = {ks_mc.statistic:.4f}')
discrim_strong = ks_mc.statistic > 0.2 and ks_mc.pvalue < 0.001
discrim_weak = ks_mc.statistic > 0.1 and ks_mc.pvalue < 0.01
if discrim_strong:
    print(f'   ✓ DISCRIMINACIÓN ESPECTRAL FUERTE')
elif discrim_weak:
    print(f'   ~ Discriminacion débil pero significativa')
else:
    print(f'   ✗ No hay discriminación espectral clara')
print()

# Veredicto compuesto
if T3_correctly and S3_correctly and (discrim_strong or discrim_weak):
    print('='*72)
    print('✓✓ EMERGENCIA GEOMÉTRICA GENUINA DETECTADA')
    print('='*72)
    print('  Cada topología se acerca a su benchmark correcto.')
    print('  T³ MC y S³ MC son espectralmente distinguibles.')
    print('  El modelo discrimina topologías dinámicamente.')
elif T3_correctly or S3_correctly:
    print('='*72)
    print('~ EMERGENCIA PARCIAL')
    print('='*72)
    print('  Algunas topologías se reconstruyen, otras no.')
else:
    print('='*72)
    print('✗ NO HAY DISCRIMINACIÓN ESPECTRAL DINÁMICA')
    print('='*72)
    print('  El MC no recupera las firmas espectrales de las topologías.')
    print('  El modelo es descriptivo pero no genera variedades.')

In [ ]:
import shutil
shutil.make_archive('plots_spec', 'zip', PLOT_DIR)
try:
    from google.colab import files
    files.download('plots_spec.zip')
except ImportError:
    pass
print('Listo. Plots en plots_spec.zip')